In [99]:
#ucitavamo potrebne biblioteke
import io, requests, pandas as pd
import matplotlib.pyplot as plt
import numpy as np
#from sklearn.mixture import GaussianMixture as GM
from scipy.stats import norm

In [ ]:
#prikupljamo skup podataka koristeci table access protocol (TAP) interfejs
#podatke dobijamo sa NASA Exoplanet Archive
TAP_URl="https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

query='''
    SELECT
    pl_orbper,
    pl_rade,
    pl_eqt,
    st_met,
    discoverymethod
    FROM pscomppars
    '''

r=requests.get(TAP_URl, params={"query": query, "format":"csv"}, timeout=120)
exo=pd.read_csv(io.StringIO(r.text))
exo.head()

Definisemo klasu vrucih Jupitera na sledeci nacin:

VJ=1 ako je orbitelni period (P) manji od 10 dana i radijus (Rp) veci 8 puta od radijusa Zemlje (Rz). U suprotnom VJ=0.

In [ ]:
#PRVI ZADATAK - udeo vrucih Jupitera u prikupljenom skupu
vj = exo[(exo['pl_orbper'] < 10) & (exo['pl_rade'] > 8)] #definisemo klasu VJ na osnovu komentara u liniji 21
proc_vj=len(vj)/len(exo) * 100 #racunanje procenta
print(f'Udeo vrucih Jupitera: {proc_vj:.4f}%') #prikazivanje rezultata

In [ ]:
#maske za vruce jupitere i metalicnost koja je veca od 0.4
m_vj=(exo["pl_orbper"] < 10) & (exo["pl_rade"] > 8)
m_met=exo["st_met"]>0.4

In [ ]:
#DRUGI ZADATAK - pokazivanje da zvezde sa [Fe/H]>0.4 imaju znacajni veci udeo VJ u odnosu na celu uzorak

visoka_metalicnost=exo[m_met] #uzimanje podataka sa visokom metalicnoscu

#racunamo udeo vrucih jupitera tako sto prvo prebrojavamo vrednosti gde vrednost ispunjava maske za VJ i metalicnost
#nakon cega ih podelimo sa maskom metalicnosti i pomnozimo sa 100
udeo_vj=(m_vj & m_met).sum() / m_met.sum() * 100

udeo_ukupno = m_vj.sum() / len(exo) * 100 #racunamo procenat vj u uzorku

print(f"Udeo vrucih Jupitera visoke metalicnosti: {udeo_vj:.4f}%")
print(f"Ukupan udeo vrucih Jupitera: {udeo_ukupno:.4f}%")

In [ ]:
#TRECI ZADATAK - primena bootstrap metode za procenu neodredjenosti uzoracke srednje vrednosti i uzoracke standardne devijacije za zvezdanu metalicnost
#koristimo se uzorkom iz 2. zad
#dobijene raspodele uporedjujemo sa analitickim izrazima

data = exo.loc[m_met, 'st_met'].dropna().values #opet izdvajamo kolonu st_met(.loc), uklanjamo prazna polja (dropna) i pravimo niz (.values)

N=15000 #broj iteracija

means=np.zeros(N) #racunamo srednju vrednost novih uzoraka koji je iste velicine kao i originalni
stds=np.zeros(N) #radimo isto kao i u gornjoj liniji, ali za standardnu devijaciju

#poredimo sa analitickim izrazima
n=len(data) #broj elemenata

#bootstrap petlja
for i in range(N):
    resample = np.random.choice(data, size=n, replace=True) #nasumicno biramo n elemenata iz data
    means[i] = np.mean(resample) #racunamo srednju vrednost
    stds[i] = np.std(resample, ddof=1) #daje nam uzoracku standardnu devijaciju

x_mean=np.mean(data) #srednja vrednost originalnog uzorka
x_std = np.std(data, ddof=1) #standardna devijacija uzorka

#analiticki izrazi
se_std = x_std / np.sqrt(2 * (n - 1))
se_mean=np.std(data,ddof=1)/np.sqrt(n)

print(f"Srednja vrednost - analiticko: {se_mean:.4f}%")
print(f"Standardna devijacija - analiticko: {se_std:.4f}")

#plotujemo grafike
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

#histogram srednjih vrednosti
axes[0].hist(means, bins=50, density=True, alpha=0.6, color='pink', label='Bootstrap raspodela')
x_vals = np.linspace(x_mean - 4*se_mean, x_mean + 4*se_mean, 200)
axes[0].plot(x_vals, norm.pdf(x_vals, x_mean, se_mean), 'r-', lw=2, label='Analiticko predvidjanje')
axes[0].set_title('Raspodela srednje vrednosti')
axes[0].set_xlabel(r'$\bar{x}$')
axes[0].legend()

#histogram standardnih devijacija
axes[1].hist(stds, bins=50, density=True, alpha=0.6, color='lightgreen', label='Bootstrap raspodela')
x_vals_s = np.linspace(x_std - 4*se_std, x_std + 4*se_std, 200)
axes[1].plot(x_vals_s, norm.pdf(x_vals_s, x_std, se_std), 'r-', lw=2, label='Analiticko predvidjanje')
axes[1].set_title('Raspodela standardne devijacije')
axes[1].set_xlabel('$s$')
axes[1].legend()

plt.tight_layout()
plt.show()

Da li je pretpostavka normalnosti jednako opravdana za srednju vrednost i standardnu devijaciju?

-> Raspodela srednje vrednosti bootstrap metodom se poklapa sa normalnom raspodelom i to potvrdjuje validnost centralne granicne teoreme.

Kod standardne devijacije primecujemo asimetricnost. Javlja se jer je standardna devijacija definisana kao koren varijanse. Samim tim osetljiva je na ekstremne uzorke koji mogu dosta da uticu na same rezultate.



Da li bi rezultat jackknife metode bili bolji od bootstrap metode?

-> U ovom slucaju jackknife bi metoda bi za srednju vrednost dala dobre rezultate koji su uporedivi sa bootstrap metodom. Kod standardne devijacije dolazi dolazi do izrazaja njena linearne priroda, tako da ne uspeva da reprodukuje varijabilnost nelinearnih statistika kao bootstrap.

Jackknife metoda nije pravi izbor pri izradi ovog zadatka.


In [ ]:
#CETVRTI ZADATAK -  imamo  primer nalazenja AIC u slucaju mesavine tri normalne raspodele. Slicnu proceduru treba da primenimo na raspodeli Rp. Koristeci GMM i AIC treba da odredimo optimalan broj komponenti M i da prikazemo fitovane raspodele. Na kraju treba da odradimo istu proceduru za discoverymethod=='Transit'.
#U oba slucaja bi trebalo da uocavamo neku prazninu.

import numpy as np
from sklearn.mixture import GaussianMixture as GM

#dato u pdf-u zadatka
rs=np.random.RandomState(seed=1)

X=np.concatenate([rs.normal(-1, 1.5, 350),
                  rs.normal(0, 1, 500),
                  rs.normal(3, 0.5, 150)]).reshape(-1, 1)

N=np.arange(1, 11)
modeli=[None for i in range(len(N))]

for i in range(len(N)):
    modeli[i]=GM(N[i]).fit(X)

AIC=[m.aic(X) for m in modeli]

najbolje_N=N[np.argmin(AIC)]
najbolji_model=modeli[np.argmin(AIC)]
print(najbolje_N, najbolji_model)

plt.plot(N, AIC, marker='o', color='yellow')
plt.xlabel('$M$')
plt.ylabel('AIC')
plt.show()

In [ ]:
#ponavljamo proceduru za raspodelu Rp
#sluzimo se primerom iz pdfa

Rp=exo['pl_rade'].dropna().values.reshape(-1, 1) #izvlacimo samo radijuse, brisemo prazne vrednosti, pretvaramo ih u niz i menjamo oblik niza

log_Rp=np.log(Rp)

N = np.arange(1, 11)
modeli = [None for i in range(len(N))]

for i in range(len(N)):
    modeli[i] = GM(N[i]).fit(log_Rp)

AIC = [m.aic(log_Rp) for m in modeli]

naj_N = N[np.argmin(AIC)]
naj_model = modeli[np.argmin(AIC)]

print("Optimalan broj komponenti:", naj_N)

In [ ]:
#plotovanje
plt.figure(figsize=(8,6))
plt.plot(N, AIC, marker='*', color='g')
plt.xlabel("Broj komponenti M")
plt.ylabel("AIC")
plt.title(r"GMM model za $R_p$")
plt.show()

In [ ]:
naj_model = GM(n_components=naj_N, random_state=42).fit(log_Rp) #fitujemo model za 2 tacke
x = np.linspace(log_Rp.min(), log_Rp.max(), 1000).reshape(-1, 1) #pravimo niz sa vise tacaka

#fitovanje raspodele
logprob = naj_model.score_samples(x)
pdf = np.exp(logprob)

#racunanje pojedinacnih komponenti
responsibilities = naj_model.predict_proba(x)
pdf_individual = responsibilities * pdf[:, np.newaxis]

#plotovanje komponenta zajedno i pojedinacno
fig, axes = plt.subplots(1, 2, figsize=(14, 6)) #kreira figuru sa 2 kolone

#prvi grafik - pojedinacno
axes[0].hist(log_Rp, bins=40, density=True, alpha=0.6, color='pink', label='Podaci')
axes[0].plot(x, pdf, '-', lw=2, label='Ukupni GMM fit', color='red')
for i in range(naj_N):
    axes[0].plot(x, pdf_individual[:, i], '--', lw=1.5, label=f'Komponenta {i+1}')
axes[0].set_title(r"Histogram $R_p$ u log skali")
axes[0].set_xlabel(r'$\log (R_p)$')
axes[0].legend()

#drugi grafik - sve zajedno
axes[1].hist(Rp, bins=80, density=True, alpha=0.6, color='orange')
axes[1].set_title(r"Histogram $R_p$ u linearnoj skali")
axes[1].set_xlabel(r'$R_p$ (Zemljini radijusi)')

plt.tight_layout()
plt.show()

In [ ]:
#radimo isti postupak za tranzite
tranzit = exo[exo["discoverymethod"] == "Transit"]
Rp_t = tranzit["pl_rade"].dropna().values.reshape(-1, 1)
log_Rp_t = np.log10(Rp_t)

In [ ]:
modeli_t = [None for i in range(len(N))]

for i in range(len(N)):
    modeli_t[i] = GM(N[i]).fit(log_Rp_t)

AIC_t = [m.aic(log_Rp_t) for m in modeli_t]

naj_N_t = N[np.argmin(AIC_t)]
naj_model_t = modeli_t[np.argmin(AIC_t)]

print("Optimalan broj komponenti:", naj_N_t)
plt.figure(figsize=(6,4))
plt.plot(N, AIC_t, marker='o', color='yellow')
plt.xlabel("Broj komponenti M")
plt.ylabel("AIC")
plt.title(r"GMM model za $R_p$ koje su detektovane metodom tranzita")
plt.show()

In [ ]:
#ponovo fitujemo model za samo 2 komponente
naj_model_t = GM(n_components=naj_N_t, random_state=42).fit(log_Rp_t)

x_t = np.linspace(log_Rp_t.min(), log_Rp_t.max(), 1000).reshape(-1, 1)

#fitovanje raspodele
logprob_t = naj_model_t.score_samples(x_t)
pdf_t = np.exp(logprob_t)

#racunanje pojedinacnih komponenti unutar GMMa
responsibilities_t = naj_model_t.predict_proba(x_t)
pdf_individual_t = responsibilities_t * pdf_t[:, np.newaxis]


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

#prvi hist - log skala
axes[0].hist(log_Rp_t, bins=80, density=True, alpha=0.6, color='purple', label='Podaci')
axes[0].plot(x_t, pdf_t, '-', lw=2, label='Ukupni GMM fit', color='red')
for i in range(naj_N_t):
    axes[0].plot(x_t, pdf_individual_t[:, i], '--', lw=1.5, label=f'Komponenta {i+1}')

axes[0].set_title(r"Histogram $R_p$ u log skali")
axes[0].set_xlabel(r'$\log (R_p)$')
axes[0].set_ylabel('Gustina verovatnoce')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

#drugi hist - linearna skala
axes[1].hist(Rp_t, bins=40, density=True, alpha=0.6, color='pink')
axes[1].set_title(r"Histogram $R_p$ u linearnoj skali")
axes[1].set_xlabel(r'$R_p$ (Zemljini radijusi)')
axes[1].set_ylabel('Gustina verovatnoce')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Posmatrajuci histograme u logaritamskoj skali, primecujemo "udubljenja", tj. Fultonovu prazninu. Ona razdvaja dve populacije egzoplaneta: manje, stenovite planete ciji su radijusi ispod 1.5R Zemlje i vece planete sa gasovitim omotacima sa radijusima iznad 2R Zemlje. Ovaj fenomen nastaje zbog procesa fotoevaporacije - visokoenergetsko zracenje maticne zvezde tokom ranih faza evolucije sistema koje dovodi do erozije gasovitih atmosfera kod planeta koje orbitiraju u blizini.

In [ ]:
#PETI ZADATAK - trazimo cija ravnotezna temperatura Teq je najbliza Teq Zemlje i za nju ispisujemo br zvezda, br planeta u tom sistemu, teleskop kojim je otkrivena,...

exo2 = exo.dropna(subset=["pl_eqt"]) #uzimamo sve za koje su određeni Teq
T_eq_z = 255
idx = (exo2["pl_eqt"] - T_eq_z).abs().idxmin()  #trazimo indeks sistema
planeta = exo2.loc[idx] #izvlaci samo sistem sa trazenim indeksom

In [ ]:
Teq = planeta["pl_eqt"] #uzimamo teq kako bi u bazi nasli taj sistem i ostale podatke koji su nam potrebni

query = f"""
SELECT
    pl_name,
    hostname,
    sy_pnum,
    sy_snum,
    pl_bmasse,
    pl_orbeccen,
    discoverymethod,
    pl_eqt,
    disc_telescope
FROM pscomppars
WHERE pl_eqt = '{Teq}'
"""

r = requests.get(TAP_URl, params={"query":query, "format": "csv"},
    timeout =120)
planeta_podaci = pd.read_csv(io.StringIO(r.text))
planeta_podaci.head()

In [ ]:
print(f'Broj zvezda: {planeta_podaci['sy_snum'].values}')
print(f'Broj planeta: {planeta_podaci["sy_pnum"].values}')
print(f'Metoda otkrica: {planeta_podaci["discoverymethod"].values}')
print(f'Masa planete: {planeta_podaci["pl_bmasse"].values}')
print(f'Ekscentricitet: {planeta_podaci["pl_orbeccen"].values}' )
print(f'Teleskop kojim je snimano je: {planeta_podaci["disc_telescope"].values}')

In [ ]:
#SESTI ZADATAK - treba da konstruisemo empirijsku kummulativnu raspodelu (ECDF) pomocu Teq planeta. Treba da primenimo metodu inverznog uzorkovanja i inverznim mapiranjem preko ECDF da dobijemo sinteticke vrednosti Ted.

#ECDF - za svaku temp nam govori koji je procenat planeta koje imaju manju ili vecu temp od te

#slicno kao i u zadacima iznad
teq=exo["pl_eqt"].dropna().values
teq_sorted=np.sort(teq) #soritranje vrendosti Teq jer ECDF mora da bude monotono rastuca fja

ecdf=np.arange(1, len(teq_sorted)+1) / len(teq_sorted) #kreiramo niz brojeva od 1 do N i delimo ga sa ukupnim brojem elemenata

#plotujemo
plt.figure(figsize=(6,4))
plt.plot(teq_sorted, ecdf, color='purple')
plt.xlabel(r"$T_{eq}$ [K]")
plt.ylabel("ECDF")
plt.title("Empirijska kumulativna raspodela")
plt.show()


In [ ]:
#metoda inverznog uzorkovanja
#generisemo N_syn uzoraka kao sto je receno u zadatku
N_syn=10**3
u=np.random.uniform(0, 1, N_syn) #generisemo 1000 nasumicnih brojeva koji su ravnomerno rasporedjeni
sinteticki_Teq = np.interp(u, ecdf, teq_sorted) #radi suprotno od ECDF

In [ ]:
#plotujemo
plt.figure(figsize=(6,4))
plt.hist(teq, bins=80, density=True, alpha=0.6, label=r"Prave vrednosti $T_{eq}$", color='purple')
plt.hist(sinteticki_Teq, bins=80, density=True, alpha=0.5, label=r"Sinteticke vrednosti$T_{eq}$", color='yellow')
plt.title("Originalna raspodela i ECDF-inverzno uzorkovanje")
plt.xlabel(r'$T_{eq}$')
plt.ylabel('Gustina verovatnoce')
plt.legend()
plt.show()

Inverzno uzorkovanje koristi ECDF kao mapu originalnih podataka. Generise nis nasumicnih verovatnoca i pomocu interpolacije pronalazi ECDF tacke na krivoj koje odgovaraju tim verovatnocama. Na taj nacin se dobija novi sinteticki uzorak koji imitira oblik i raspodelu originalnog skupa podataka.

Bootstrap metoda generise nove uzorke na osnovu originalnog skupa podataka sa vracanjem (eng. resampling with replacement). Ova metoda stvara vise podskupova iste velicine kao original i tako vrsi procenu varijabilnosti.